<a href="https://colab.research.google.com/github/thasithgajanayake/Telco-Customer-Churn/blob/main/Telco_Churn_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# IMPORTS, DATA LOADING AND DATA CLEANING

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

# Set visual theme and create output folder for figures
sns.set_theme(style="whitegrid")
os.makedirs('figures', exist_ok=True)

# 1. Load dataset (ensure filename matches your uploaded CSV)
df = pd.read_csv('Telco-Customer-Churn.csv')

print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Column Info ---")
df.info()

# 2. Clean TotalCharges (convert blank strings to NaN and impute with 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"\nMissing values in TotalCharges before imputation: {df['TotalCharges'].isnull().sum()}")
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 3. Check target class distribution
print("\n--- Churn Class Distribution (%) ---")
print(df['Churn'].value_counts(normalize=True) * 100)

--- Dataset Shape ---
(7043, 21)

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   o

In [ ]:
# TASK 1 - EXPLORATORY DATA ANALYSIS & VISUALIZATION

In [ ]:
# Figure 1: Target Variable Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Churn', hue='Churn', palette='Set2', legend=False)
plt.title('Distribution of Customer Churn', fontsize=12)
plt.xlabel('Churn Status')
plt.ylabel('Count')

total = len(df)
for p in ax.patches:
    pct = f'{100 * p.get_height() / total:.1f}%\n({int(p.get_height())})'
    ax.annotate(pct, (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', weight='bold')

plt.tight_layout()
plt.savefig('figures/01_churn_distribution.png', dpi=300)
plt.show()

In [ ]:
# Figure 2: Numerical Feature Distributions by Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, palette='Set2', ax=axes[0])
axes[0].set_title('Tenure Distribution by Churn')
axes[0].set_xlabel('Tenure (Months)')

sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, fill=True, palette='Set2', ax=axes[1])
axes[1].set_title('Monthly Charges Distribution by Churn')
axes[1].set_xlabel('Monthly Charges ($)')

plt.tight_layout()
plt.savefig('figures/02_numerical_distributions.png', dpi=300)
plt.show()

In [ ]:
# Figure 3: Categorical Churn Drivers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2', ax=axes[0])
axes[0].set_title('Churn by Contract Type')
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=df, x='InternetService', hue='Churn', palette='Set2', ax=axes[1])
axes[1].set_title('Churn by Internet Service')

sns.countplot(data=df, x='TechSupport', hue='Churn', palette='Set2', ax=axes[2])
axes[2].set_title('Churn by Tech Support')

plt.tight_layout()
plt.savefig('figures/03_categorical_drivers.png', dpi=300)
plt.show()

In [ ]:
# Figure 4: Correlation Heatmap
plt.figure(figsize=(7, 5))
numeric_df = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
numeric_df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Heatmap with Churn')
plt.tight_layout()
plt.savefig('figures/04_correlation_heatmap.png', dpi=300)
plt.show()